In [1]:
from google.colab import drive
drive.mount('/content/drive', force_remount=True)
!ln -s /content/drive/MyDrive/Colab\ Notebooks/SES-Inference /mydrive

Mounted at /content/drive


In [2]:
import glob
import os
import numpy as np
import pandas as pd

In [65]:
import sys
sys.path.append('/mydrive/libs')

%load_ext autoreload
%autoreload 2

from utils.constants import PPLACE_RURAL

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [66]:
countries = ['Sierra Leone','Uganda']
dhsloc = 'rc'

In [72]:
cols = ['country','years','households','clusters','clusters_urban','clusters_rural','mean_iwi','std_iwi','min_iwi','max_iwi','pplaces','pplaces_urban','pplaces_rural','swapped']
df = pd.DataFrame(columns=cols)
for country in countries:
  
  fn_swap = glob.glob(f"/mydrive/data/{country}/results/features/*{dhsloc}_cluster_pplace_ids.csv")[0]
  tmp_swap = pd.read_csv(fn_swap,index_col=0) 
  
  fn_hh = glob.glob(f"/mydrive/data/{country}/results/features/households/*_iwi_household.csv")[0]
  tmp_hh = pd.read_csv(fn_hh,index_col=0)

  fn_cc = glob.glob(f"/mydrive/data/{country}/results/features/clusters/*_iwi_cluster.csv")[0] 
  tmp_cc = pd.read_csv(fn_cc,index_col=0)

  fn_pp = glob.glob(f"/mydrive/data/{country}/results/features/pplaces/PPLACES.csv")[0]
  tmp_pp = pd.read_csv(fn_pp,index_col=0)

  obj = dict(country=country,
             years=",".join(tmp_swap.dhs_year.unique().astype(str)),
             households=tmp_hh.shape[0],
             clusters=tmp_swap.shape[0],
             clusters_urban=tmp_swap.query("dhs_rural==0").shape[0],
             clusters_rural=tmp_swap.query("dhs_rural==1").shape[0],
             mean_iwi=tmp_swap.dhs_mean_iwi.mean(),
             std_iwi=tmp_swap.dhs_std_iwi.mean(),
             min_iwi=tmp_swap.dhs_mean_iwi.min(),
             max_iwi=tmp_swap.dhs_mean_iwi.max(),
             pplaces=tmp_pp.shape[0],
             pplaces_urban=tmp_pp.query("place not in @PPLACE_RURAL").shape[0],
             pplaces_rural=tmp_pp.query("place in @PPLACE_RURAL").shape[0],
             swapped=tmp_swap.query("OSMID>0").shape[0])
  
  df = df.append(pd.DataFrame(obj, index=[0], columns=cols)[cols], ignore_index=True)  

In [81]:
numeric_c = []
for c in df.columns:
  if c not in ['country','years']:
    numeric_c.append(c)
    df.loc[:,c] = df.loc[:,c].astype(np.float32).round(1)
df

,country,years,households,clusters,clusters_urban,clusters_rural,mean_iwi,std_iwi,min_iwi,max_iwi,pplaces,pplaces_urban,pplaces_rural,swapped
0,Sierra Leone,"2016,2019",19975.0,893.0,308.0,585.0,26.200001,10.4,4.2,72.699997,9568.0,64.0,9504.0,585.0
1,Uganda,"2016,2018",27798.0,1001.0,242.0,759.0,24.799999,11.4,2.0,82.900002,13902.0,202.0,13700.0,714.0


In [82]:
numeric_c

['households',
 'clusters',
 'clusters_urban',
 'clusters_rural',
 'mean_iwi',
 'std_iwi',
 'min_iwi',
 'max_iwi',
 'pplaces',
 'pplaces_urban',
 'pplaces_rural',
 'swapped']

In [83]:
df.pivot_table(columns='country').loc[numeric_c,:]

country,Sierra Leone,Uganda
households,19975.000000,27798.000000
clusters,893.000000,1001.000000
clusters_urban,308.000000,242.000000
clusters_rural,585.000000,759.000000
mean_iwi,26.200001,24.799999
std_iwi,10.400000,11.400000
min_iwi,4.200000,2.000000
max_iwi,72.699997,82.900002
pplaces,9568.000000,13902.000000
pplaces_urban,64.000000,202.000000
